<a href="https://colab.research.google.com/github/financieras/antigen_predictor/blob/main/notebooks/01_dataset_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01: Exploración del Dataset IEDB

El objetivo de este notebook es entender la estructura del archivo de exportación
completo de IEDB antes de construir nuestro dataset de entrenamiento.

No construimos nada aquí. Solo miramos, filtramos y documentamos lo que encontramos.

Al final del notebook sabremos:
- Qué columnas son relevantes para nuestro proyecto
- Cuántos epítopos hay de SARS-CoV-2 e Influenza A
- Qué proporción de ensayos son positivos vs negativos
- Qué columna identifica la proteína fuente de cada epítopo

In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/epitope/epitope_full_v3.csv'
LOCAL_PATH = '/content/epitope_full_v3.csv'

if not os.path.exists(LOCAL_PATH):
    print("📥 Copiando archivo desde Drive...")
    !cp "{DRIVE_PATH}" "{LOCAL_PATH}"
    print("✅ Copia terminada.")

print("Cargando dataset...")
df = pd.read_csv(LOCAL_PATH, low_memory=False)
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargando dataset...
Dataset cargado: 2,421,966 filas × 32 columnas


## 0. Configuración y montaje de Google Drive

In [ ]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

Mounted at /content/drive


## 1. Carga del dataset

El archivo completo pesa ~830 MB. La carga puede tardar 1-2 minutos.  
Usamos `low_memory=False` para evitar warnings de tipos mixtos en columnas grandes.

In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/epitope/epitope_full_v3.csv'
LOCAL_PATH = '/content/epitope_full_v3.csv'

if not os.path.exists(LOCAL_PATH):
    print("📥 Copiando archivo desde Drive...")
    !cp "{DRIVE_PATH}" "{LOCAL_PATH}"
    print("✅ Copia terminada.")

print("Cargando dataset...")
df = pd.read_csv(LOCAL_PATH, low_memory=False)
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Cargando dataset... (puede tardar un par de minutos)
Dataset cargado: 2,421,966 filas × 32 columnas


## 2. Primera inspección: estructura general

In [ ]:
# Primeras filas
df.head()

,Epitope ID,Epitope,Epitope.1,Epitope.2,Epitope.3,Epitope.4,Epitope.5,Epitope.6,Epitope.7,Epitope.8,...,Related Object.5,Related Object.6,Related Object.7,Related Object.8,Related Object.9,Related Object.10,Related Object.11,Related Object.12,Related Object.13,Related Object.14
0,IEDB IRI,Object Type,Name,Modified Residue(s),Modifications,Starting Position,Ending Position,IRI,Synonyms,Source Molecule,...,IRI,Synonyms,Source Molecule,Source Molecule IRI,Molecule Parent,Molecule Parent IRI,Source Organism,Source Organism IRI,Species,Species IRI
1,http://www.iedb.org/epitope/1,Linear peptide,"AA + MCM(A1,A2)","A1,A2",Main chain modification,200,201,NaN,NaN,"streptokinase, SKase",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,http://www.iedb.org/epitope/2,Linear peptide,AAAAAAAAAAAAA,NaN,NaN,489,501,NaN,NaN,RNA-binding protein 47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,http://www.iedb.org/epitope/3,Linear peptide,AAAAAAAAAAAANANIAAAA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,lpqH,Lipoprotein lpqH precursor,http://www.ncbi.nlm.nih.gov/protein/P0A5J0.1,Lipoprotein LpqH,http://www.uniprot.org/uniprot/P9WK61,Mycobacterium tuberculosis,http://purl.obolibrary.org/obo/NCBITaxon_1773,Mycobacterium tuberculosis,http://purl.obolibrary.org/obo/NCBITaxon_1773
4,http://www.iedb.org/epitope/4,Linear peptide,AAAAAAAAAAAGNVNIAAAA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,lpqH,Lipoprotein lpqH precursor,http://www.ncbi.nlm.nih.gov/protein/P0A5J0.1,Lipoprotein LpqH,http://www.uniprot.org/uniprot/P9WK61,Mycobacterium tuberculosis,http://purl.obolibrary.org/obo/NCBITaxon_1773,Mycobacterium tuberculosis,http://purl.obolibrary.org/obo/NCBITaxon_1773


In [ ]:
# Todas las columnas disponibles
print(f"Total de columnas: {len(df.columns)}\n")
for i, col in enumerate(df.columns):
    print(f"  {i:>3}. {col}")

Total de columnas: 32

    0. Epitope ID
    1. Epitope
    2. Epitope.1
    3. Epitope.2
    4. Epitope.3
    5. Epitope.4
    6. Epitope.5
    7. Epitope.6
    8. Epitope.7
    9. Epitope.8
   10. Epitope.9
   11. Epitope.10
   12. Epitope.11
   13. Epitope.12
   14. Epitope.13
   15. Epitope.14
   16. Epitope.15
   17. Related Object
   18. Related Object.1
   19. Related Object.2
   20. Related Object.3
   21. Related Object.4
   22. Related Object.5
   23. Related Object.6
   24. Related Object.7
   25. Related Object.8
   26. Related Object.9
   27. Related Object.10
   28. Related Object.11
   29. Related Object.12
   30. Related Object.13
   31. Related Object.14


In [ ]:
# Tipos de datos y valores no nulos por columna
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2421966 entries, 0 to 2421965
Data columns (total 32 columns):
 #   Column             Non-Null Count    Dtype 
---  ------             --------------    ----- 
 0   Epitope ID         2421966 non-null  object
 1   Epitope            2421966 non-null  object
 2   Epitope.1          2421966 non-null  object
 3   Epitope.2          180042 non-null   object
 4   Epitope.3          180042 non-null   object
 5   Epitope.4          2069372 non-null  object
 6   Epitope.5          2069372 non-null  object
 7   Epitope.6          3714 non-null     object
 8   Epitope.7          3556 non-null     object
 9   Epitope.8          2226960 non-null  object
 10  Epitope.9          2221506 non-null  object
 11  Epitope.10         2046056 non-null  object
 12  Epitope.11         2045732 non-null  object
 13  Epitope.12         2228340 non-null  object
 14  Epitope.13         2187881 non-null  object
 15  Epitope.14         2228340 non-null  object
 16  

## 3. Identificar las columnas clave para nuestro proyecto

Para construir nuestro dataset necesitamos al menos:
- La **proteína fuente** del epítopo (para saber a qué proteína pertenece)
- El **organismo** fuente (para filtrar por SARS-CoV-2 e Influenza A)
- El **resultado del ensayo** (positivo o negativo)

Buscamos esas columnas por nombre aproximado.

In [ ]:
# Buscar columnas relacionadas con el organismo fuente
organism_cols = [c for c in df.columns if 'organism' in c.lower() or 'host' in c.lower()]
print("Columnas relacionadas con organismo:")
for c in organism_cols:
    print(f"  - {c}")

In [ ]:
# Buscar columnas relacionadas con la proteína fuente
protein_cols = [c for c in df.columns if 'protein' in c.lower() or 'antigen' in c.lower() or 'source' in c.lower()]
print("Columnas relacionadas con proteína/antígeno fuente:")
for c in protein_cols:
    print(f"  - {c}")

In [ ]:
# Buscar columnas relacionadas con el resultado del ensayo
assay_cols = [c for c in df.columns if 'qualitative' in c.lower() or 'outcome' in c.lower() or 'assay' in c.lower() or 'result' in c.lower()]
print("Columnas relacionadas con resultado de ensayo:")
for c in assay_cols:
    print(f"  - {c}")

## 4. Verificar las columnas clave

Una vez identificadas las columnas relevantes, verificamos que existen con los nombres esperados.  
**Ajusta los nombres si las celdas anteriores muestran nombres distintos.**

In [ ]:
# Columnas que usaremos — ajustar si es necesario tras ver los resultados anteriores
COL_ORGANISM   = 'Epitope Source Organism Name'   # nombre del organismo fuente
COL_PROTEIN    = 'Epitope Source Molecule Name'   # nombre de la proteína fuente
COL_PROTEIN_ID = 'Epitope Source Molecule IRI'    # identificador UniProt de la proteína
COL_ASSAY      = 'Assay Qualitative Measure'      # resultado del ensayo

# Verificar que existen
print("Verificando columnas clave:")
for col in [COL_ORGANISM, COL_PROTEIN, COL_PROTEIN_ID, COL_ASSAY]:
    exists = col in df.columns
    print(f"  {'✓' if exists else '✗'} '{col}'")

In [ ]:
# Valores únicos del campo de resultado de ensayo
print("Valores únicos en resultado de ensayo:")
print(df[COL_ASSAY].value_counts(dropna=False))

## 5. Filtrar por organismos de interés

Buscamos filas relacionadas con SARS-CoV-2 e Influenza A.  
Primero miramos qué strings exactos usa IEDB para nombrarlos.

In [ ]:
# Ver los 30 organismos más frecuentes para identificar los nuestros
print("Top 30 organismos más frecuentes en el dataset:")
print(df[COL_ORGANISM].value_counts().head(30))

In [ ]:
# Filtrar por SARS-CoV-2 e Influenza A usando búsqueda parcial
mask_sars = df[COL_ORGANISM].str.contains(
    'SARS-CoV-2|Severe acute respiratory syndrome coronavirus 2',
    case=False, na=False
)
mask_flu = df[COL_ORGANISM].str.contains(
    'Influenza A',
    case=False, na=False
)

df_sars = df[mask_sars]
df_flu  = df[mask_flu]

print(f"Filas SARS-CoV-2:  {len(df_sars):>8,}")
print(f"Filas Influenza A: {len(df_flu):>8,}")
print(f"Total combinado:   {len(df_sars) + len(df_flu):>8,}")

## 6. Distribución de ensayos positivos vs negativos

Nuestro modelo necesita ejemplos positivos (antigénicos) y negativos.  
Veamos cuántos hay de cada tipo en los organismos de interés.

In [ ]:
df_targets = pd.concat([df_sars, df_flu], ignore_index=True)

print("Distribución de resultados de ensayo en nuestros organismos:")
print(df_targets[COL_ASSAY].value_counts(dropna=False))
print(f"\nTotal de filas: {len(df_targets):,}")

In [ ]:
# Solo ensayos positivos
df_positive = df_targets[df_targets[COL_ASSAY] == 'Positive']
print(f"Epítopos con ensayo positivo: {len(df_positive):,}")

# Proteínas únicas con epítopos positivos
n_proteins = df_positive[COL_PROTEIN].nunique()
print(f"Proteínas fuente únicas:      {n_proteins:,}")

## 7. Explorar las proteínas fuente más representadas

Queremos saber qué proteínas tienen más epítopos validados.  
Esto nos da una idea de qué veremos en la demo final (Spike debería estar arriba).

In [ ]:
print("Proteínas de SARS-CoV-2 con más epítopos positivos:")
sars_positive = df_sars[df_sars[COL_ASSAY] == 'Positive']
print(sars_positive[COL_PROTEIN].value_counts().head(15))

In [ ]:
print("Proteínas de Influenza A con más epítopos positivos:")
flu_positive = df_flu[df_flu[COL_ASSAY] == 'Positive']
print(flu_positive[COL_PROTEIN].value_counts().head(15))

## 8. Inspección de nulos en columnas clave

Antes de guardar el subconjunto filtrado, comprobamos cuántos valores nulos
hay en las columnas que usaremos en el siguiente notebook.

In [ ]:
cols_to_check = [COL_ORGANISM, COL_PROTEIN, COL_PROTEIN_ID, COL_ASSAY]
print("Valores nulos en columnas clave (sobre el subconjunto filtrado):")
print(df_targets[cols_to_check].isnull().sum())
print(f"\nTotal filas: {len(df_targets):,}")

## 9. Guardar el subconjunto filtrado

Guardamos solo las filas de nuestros dos organismos para no tener que
volver a cargar el archivo de 830 MB en el Notebook 02.

In [ ]:
OUTPUT_PATH = '/content/drive/MyDrive/epitope/iedb_sars_flu_filtered.csv'

df_targets.to_csv(OUTPUT_PATH, index=False)
print(f"Archivo guardado en: {OUTPUT_PATH}")
print(f"Tamaño: {len(df_targets):,} filas × {df_targets.shape[1]} columnas")

## 10. Resumen de hallazgos

Completa esta celda después de ejecutar el notebook con los valores reales.

| Concepto | Valor |
|---|---|
| Total de filas en IEDB | |
| Filas de SARS-CoV-2 | |
| Filas de Influenza A | |
| Epítopos positivos (combinado) | |
| Proteínas únicas con epítopos positivos | |
| Proteína más representada en SARS-CoV-2 | |
| Proteína más representada en Influenza A | |

### Columnas clave confirmadas
- Organismo fuente: `Epitope Source Organism Name`
- Proteína fuente: `Epitope Source Molecule Name`
- ID de proteína: `Epitope Source Molecule IRI`
- Resultado de ensayo: `Assay Qualitative Measure`

### Decisiones para el Notebook 02
- Usaremos `label = 1` para proteínas con al menos un epítopo positivo en IEDB
- Usaremos `label = 0` para proteínas del mismo patógeno sin epítopos positivos
- Archivo de entrada para NB02: `iedb_sars_flu_filtered.csv`